In [1]:
!pip install pyspark

In [2]:
from google.colab import files
uploaded = files.upload()

Saving q1_dataset(1).txt to q1_dataset(1).txt


In [3]:
import re
from pyspark import SparkContext

# Initialize Spark Context
sc = SparkContext.getOrCreate()

# Load the dataset from the uploaded file
input_file = "q1_dataset(1).txt"
lines = sc.textFile(input_file)

# Part A: Character count
# We split each line into a list of characters, filter out whitespace, map them to (char, 1) pairs, and reduce by key to get totals
char_counts = lines.flatMap(lambda line: list(line)) \
                   .filter(lambda char: not char.isspace()) \
                   .map(lambda char: (char, 1)) \
                   .reduceByKey(lambda a, b: a + b)
print("Part A: Character Counts (Sample)")
print(char_counts.take(10))

# Part B: Word count for specific words
target_words = {"creature", "death", "nature"}
# Preprocessing: Convert to lowercase and extract words (removing punctuation)
def preprocess_words(line):
    # Using regex to find words consisting only of letters
    return re.findall(r'[a-z]+', line.lower())
specific_word_counts = lines.flatMap(preprocess_words) \
                            .filter(lambda word: word in target_words) \
                            .map(lambda word: (word, 1)) \
                            .reduceByKey(lambda a, b: a + b)
print("\n Part B: Specific Word Counts")
print(specific_word_counts.collect())

# Part C: Top 20 Most Frequent Bigrams
def extract_bigrams(line):
    # Preprocessing: lowercase and extract words
    words = re.findall(r'[a-z]+', line.lower())
    # Create bigram pairs only if there are at least two words in the line
    # This keeps each line independent
    return [((words[i], words[i+1]), 1) for i in range(len(words)-1)]
top_20_bigrams = lines.flatMap(extract_bigrams) \
                      .reduceByKey(lambda a, b: a + b) \
                      .takeOrdered(20, key=lambda x: -x[1])
print("\n Part C: Top 20 Most Frequent Bigrams")
for bigram, count in top_20_bigrams:
    print(f"{bigram[0]} {bigram[1]}: {count}")

Part A: Character Counts (Sample)
[('o', 25081), ('j', 431), ('c', 9060), ('t', 29724), ('G', 201), ('b', 4766), ('g', 5779), ('F', 215), (';', 976), ('O', 173)]

 Part B: Specific Word Counts
[('nature', 53), ('death', 79), ('creature', 44)]

 Part C: Top 20 Most Frequent Bigrams
of the: 533
of my: 264
in the: 263
i was: 214
i had: 207
that i: 198
to the: 197
and i: 192
and the: 185
which i: 145
on the: 143
it was: 135
by the: 128
but i: 123
when i: 122
in my: 122
of a: 120
to me: 118
i am: 110
with the: 104
